In [1]:
import sys
import os
sys.path.append('../TCT/')
import TCT
from TCT import translator_kpinfo
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_query
from TCT import TCT_pathfinder
import time
import json

In [2]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources()

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


In [7]:
for api in list(API_predicates.keys()):
    print(api)
    print(API_predicates[api])

Text Mined Cooccurrence API
['biolink:occurs_together_in_literature_with']
Automat-hgnc(Trapi v1.5.0)
['biolink:has_part']
Automat-gtex(Trapi v1.5.0)
['biolink:is_splice_site_variant_of', 'biolink:is_nearby_variant_of', 'biolink:related_to', 'biolink:is_synonymous_variant_of', 'biolink:is_missense_variant_of', 'biolink:is_nonsense_variant_of', 'biolink:is_non_coding_variant_of', 'biolink:affects', 'biolink:is_frameshift_variant_of']
Automat-hetionet(Trapi v1.5.0)
['biolink:directly_physically_interacts_with', 'biolink:regulates', 'biolink:correlated_with', 'biolink:has_part', 'biolink:similar_to', 'biolink:related_to', 'biolink:ameliorates_condition', 'biolink:actively_involved_in', 'biolink:genetically_interacts_with', 'biolink:subclass_of', 'biolink:catalyzes', 'biolink:expressed_in', 'biolink:causes', 'biolink:treats']
Automat-cam-kp(Trapi v1.5.0)
['biolink:acts_upstream_of_or_within_negative_effect', 'biolink:directly_physically_interacts_with', 'biolink:has_phenotype', 'biolink:ha

In [3]:
# select a list of APIs to use and a list of predicates to use
selected_APIlist = []

if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)
print(selected_metaKG.shape)

(22568, 5)


In [4]:
subject_name = 'NFKB1'
subject_node = 'NCBIGene:4790'
subject_category = ['biolink:Gene', 'biolink:Protein']
name_resolver.lookup(subject_name, return_top_response=False, taxon_id=9606)

[TranslatorNode(curie='NCBIGene:442859', label='NFKB1', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9615']),
 TranslatorNode(curie='NCBIGene:114036698', label='NFKB1', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin',

In [7]:
object_name = 'T-cell'
#object_node = 'NCBIGene:7157'
object_category = ['biolink:CellType']
name_resolver.lookup(object_name, return_top_response=False)


[TranslatorNode(curie='MONDO:0004977', label='angioimmunoblastic T-cell lymphoma', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0019468', label='T-cell prolymphocytic leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0015760', label='T-Cell Lymphoma', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='UniProtKB:A0A4X1THI7', label='A0A4X1THI7_PIG T cell activation inhibitor, mitochondri

In [4]:
intermediate_categories = ['biolink:Gene','biolink:Protein']


In [19]:
pathfinder_query_json = TCT_pathfinder.format_query_json_for_pathfinder_with_constraints(subject_ids=[subject_node], 
                                                   object_ids=[object_node], 
                                                   constraints_intermediate_category=intermediate_categories)

In [ ]:
# TCT pathfinder pipeline
start_time = time.time()
result1, result2, TCT_path_finder_result = TCT_pathfinder.pathfinder(input_node1_id=subject_node, input_node2_id= object_node, #COVID-19
                                                                            intermediate_categories=intermediate_categories, 
                                                                            APInames=select_APIs, 
                                                                            metaKG=selected_metaKG, 
                                                                            API_predicates=API_predicates, 
                                                                            scoring_method='infores')


end_time = time.time()

TCT_execution_time = end_time - start_time

# return results path_finder_result to a json file
import json
with open(f'TCT_path_finder_result__{subject_name.replace(":", "_")}__{object_name.replace(":", "_")}.json', 'w') as f:
    json.dump(TCT_path_finder_result, f, indent=4)
Number_of_paths_TCT = len(TCT_path_finder_result['auxiliary_graphs'])
print(f"Number of paths TCT: {Number_of_paths_TCT}")

NCBIGene:7157
CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!
Automat-cam-kp(Trapi v1.5.0): Success!
Automat-genome-alliance(Trapi v1.5.0): Success!
Automat-hetionet(Trapi v1.5.0): Success!
RTX KG2 - TRAPI 1.5.0: Success!
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
Automat-robokop(Trapi v1.5.0): Success!
BioThings Explorer (BTE) TRAPI: Success!
CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!


In [ ]:
# query arax pathfinder with constraints and write response to a json file
start_time = time.time()
result_arax = TCT_pathfinder.query_arax_pathfinder_with_constraints(subject_node, 'biolink:Drug', object_node, 'biolink:Disease', constraints=intermediate_categories)
end_time = time.time()
arax_execution_time = end_time - start_time
if result_arax.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_ARAX_constrained = len(result_arax.json()['message']['auxiliary_graphs'])
else:    
    Number_of_paths_ARAX_constrained = 0
print(f"ARAX execution time with constraints: {arax_execution_time} seconds")
print(f"Number of paths ARAX with constraints: {Number_of_paths_ARAX_constrained}")
import json
with open('arax_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'with_constraints.json', 'w') as f:
    json.dump(result_arax.json()['message'], f, indent=4)


In [ ]:
# query aragorn pathfinder with constraints and write response to a json file
start_time = time.time()
result_aragorn = TCT_pathfinder.query_aragorn_pathfinder_with_constraints(subject_node, 'biolink:Drug', object_node, 'biolink:Disease', constraints=intermediate_categories)
end_time = time.time()
aragorn_execution_time = end_time - start_time
if result_aragorn.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_ARAGORN_constrained = len(result_aragorn.json()['message']['auxiliary_graphs'])
else:    
    Number_of_paths_ARAGORN_constrained = 0
print(f"ARAGORN execution time with constraints: {aragorn_execution_time} seconds")
print(f"Number of paths ARAGORN with constraints: {Number_of_paths_ARAGORN_constrained}")
import json
with open('aragorn_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'with_constraints.json', 'w') as f:
    json.dump(result_aragorn.json()['message'], f, indent=4)
    

In [ ]:
# run aragorn pathfinder without constraints and write response to a json file
start_time = time.time()
aragorn_response = TCT_pathfinder.query_aragorn_pathfinder(subject_node, 
                                                           subject_category, 
                                                           object_node, 
                                                           object_category)
# write response to a json file
import json
if 'message' in aragorn_response.json():
    with open('aragorn_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'.json', 'w') as f:
        json.dump(aragorn_response.json()['message'], f, indent=4)
end_time = time.time()
aragorn_execution_time = end_time - start_time
Number_of_paths_aragorn = len(aragorn_response.json()['message']['auxiliary_graphs'])
print(f"Aragorn execution time: {aragorn_execution_time} seconds")
print(f"Number of paths Aragorn: {Number_of_paths_aragorn}")

Aragorn execution time: 116.52554869651794 seconds
Number of paths Aragorn: 1000


In [ ]:
# run arax pathfinder without constraints and write response to a json file
start_time = time.time()
arax_response = TCT_pathfinder.query_arax_pathfinder(subject_node, 'biolink:Drug', object_node, 'biolink:Disease')
# write response to a json file
import json
with open('arax_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'.json', 'w') as f:
    json.dump(arax_response.json()['message'], f, indent=4)
end_time = time.time()
arax_execution_time = end_time - start_time
if arax_response.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_ARAX = len(arax_response.json()['message']['auxiliary_graphs'])
else:
    Number_of_paths_ARAX = 0
print(f"ARAX execution time: {arax_execution_time} seconds")
print(f"Number of paths ARAX: {Number_of_paths_ARAX}")

ARAX execution time: 122.5346007347107 seconds
Number of paths ARAX: 500


ARAX execution time with constraints: 117.56282091140747 seconds
Number of paths ARAX with constraints: 500


ARAGORN execution time with constraints: 105.8430609703064 seconds
Number of paths ARAGORN with constraints: 1000
